# 7장. NLP 개요와 데이터셋

| 항목 | 내용 |
|---|---|
| 예상 시간 | 3시간 |
| 선수 장 | 5장 |
| 이 장의 결과물 | 영어·한국어 데이터 로딩 함수 (`src/data.py`), 데이터 점검 결과 |
| 과제 연결 | 데이터 준비: 500건 이상, 2개 이상 카테고리, 결측 제거, 개인정보·라이선스 확인 |

**학습 목표**
- 코퍼스, 문서, 토큰, 어휘, 레이블, OOV를 구분해서 말할 수 있다.
- 20 Newsgroups에서 헤더·서명·인용문을 지워야 하는 이유를 실험 결과로 설명할 수 있다.
- KLUE-YNAT의 구조와 레이블이 만들어진 방식을 알고, 주제별로 균등하게 뽑을 수 있다.
- 과제의 데이터 체크리스트를 코드로 점검할 수 있다.

## 7.1 자연어 처리(NLP)는 어디에 쓰이나

자연어 처리(Natural Language Processing)는 사람이 쓰는 언어를 컴퓨터가 다루게 하는 기술입니다.

| 서비스 | 하는 일 | 이 과제와의 관계 |
|---|---|---|
| 검색 엔진 | 질문과 비슷한 문서를 찾아 순위대로 보여 준다 | **정보 검색**: 15–16장 |
| 뉴스 분류, 스팸 필터 | 문서를 정해진 카테고리로 나눈다 | **텍스트 분류**: 17–20장 |
| 추천 시스템 | 읽은 글과 비슷한 글을 추천한다 | 검색과 같은 유사도 계산 |

두 작업 모두 출발점이 같습니다. **글자를 숫자 벡터로 바꾸는 것**입니다. 이 과제는 그중 가장 고전적인 방법인 **BoW(Bag of Words)와 TF-IDF**를 직접 구현합니다.

## 7.2 기본 용어

In [2]:
from collections import Counter

MINI_DOCS = [
    "The rocket launched into space!",
    "NASA launched a new space telescope.",
    "The pitcher threw a fast ball.",
    "Our team won the baseball game.",
    "Astronauts watched the baseball game in space.",
]
tokens = [d.lower().strip(".!").split() for d in MINI_DOCS]
all_tokens = [t for doc in tokens for t in doc]

print("문서 수            :", len(MINI_DOCS))
print("토큰 수 (전체 단어):", len(all_tokens))
print("어휘 수 (다른 단어):", len(set(all_tokens)))
print("가장 흔한 단어      :", Counter(all_tokens).most_common(3))

문서 수            : 5
토큰 수 (전체 단어): 30
어휘 수 (다른 단어): 21
가장 흔한 단어      : [('the', 4), ('space', 3), ('launched', 2)]


| 용어 | 뜻 | 미니 코퍼스에서 |
|---|---|---|
| **코퍼스(corpus)** | 분석 대상 문서 전체 | 문서 5개 |
| **문서(document)** | 검색·분류의 단위 하나. 게시글, 기사 제목, 리뷰 등 | `"The rocket launched into space!"` |
| **토큰(token)** | 문서를 쪼갠 단위 하나. 같은 단어가 두 번 나오면 토큰 2개 | 전체 30개 |
| **어휘(vocabulary)** | 서로 다른 토큰의 목록. 벡터의 차원(열 수)이 된다 | 21개 |
| **레이블(label)** | 문서의 정답 카테고리 | `space`, `baseball` |
| **OOV(Out-Of-Vocabulary)** | 학습 어휘에 없는 토큰. 벡터에 반영되지 않는다 | 쿼리 `"moon"` |

토큰 수와 어휘 수의 차이를 기억하세요. `the`는 토큰으로는 4개지만 어휘로는 1개입니다.

## 7.3 영어 데이터: 20 Newsgroups

1990년대 유즈넷 뉴스그룹(게시판) 20곳의 글 약 18,000건입니다. scikit-learn에 내장되어 바로 쓸 수 있습니다. 이 과제에서는 그중 **5개 게시판**을 씁니다.

| 카테고리 | 주제 | 고른 이유 |
|---|---|---|
| `comp.graphics` | 컴퓨터 그래픽스 | `comp.windows.x`와 **비슷한 쌍** (둘 다 컴퓨터) |
| `comp.windows.x` | X 윈도 시스템 | |
| `rec.sport.baseball` | 야구 | `rec.sport.hockey`와 **비슷한 쌍** (둘 다 스포츠) |
| `rec.sport.hockey` | 하키 | |
| `sci.space` | 우주 | 나머지와 **다른 주제** |

쉬운 구분(우주 vs 스포츠)과 어려운 구분(야구 vs 하키)이 섞여 있어서, 분류 결과에서 BoW가 어디서 헷갈리는지 보기 좋습니다.

### 원본 글의 구조
먼저 아무것도 지우지 않은 원본을 봅니다. 출력에서 이메일 주소와 작성자 정보 줄은 가립니다(7.5절에서 이유를 다룹니다).

In [3]:
import re

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
RANDOM_STATE = 42

raw_bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES, remove=(),
                               shuffle=True, random_state=RANDOM_STATE)
EMAIL = re.compile(r"\S+@\S+\.\S+")


def redact(text: str, max_lines: int = 18) -> str:
    """출력용: 작성자 정보 줄과 인용 출처 줄을 가리고 이메일을 지운다."""
    out = []
    for line in text.splitlines()[:max_lines]:
        if re.match(r"(From|Reply-To|Sender):", line) or re.search(r"(writes|wrote):\s*$", line):
            out.append("<작성자 정보 가림>")
        else:
            out.append(EMAIL.sub("<email>", line))
    return "\n".join(out)


# 인용문(>)이 있는 글을 하나 골라 본다
idx = next(i for i, t in enumerate(raw_bunch.data) if "\n>" in t and len(t) < 1500)
print("레이블:", raw_bunch.target_names[raw_bunch.target[idx]])
print("-" * 60)
print(redact(raw_bunch.data[idx]))

레이블: rec.sport.hockey
------------------------------------------------------------
<작성자 정보 가림>
Subject: Re: Bruins vs Canadiens:
Organization: University of Waterloo
Lines: 24

<작성자 정보 가림>
<작성자 정보 가림>
>
>or Boston? You know Sinden's going to find some way of screwing up 
>even this good Boston team. He'll fire Suter or trade away a vital 
>star. (Admittedly, his last few trades have been good ones but how 
>long before his luck runs out and he starts making Esposito-for-
>Ratelle type trades again?) 
>

How was this trade bad? I seem to recall Ratelle and Middleton making a pretty
good centre - right wing combination, and the Bruins also got Brad Park in the
deal (and they also lost Vadnais and somebody else). After the trade, the 


게시글은 세 부분으로 되어 있습니다.
- **헤더(headers)**: 맨 위 `이름: 값` 줄들. 작성자, 제목(Subject), 소속(Organization), 줄 수 등
- **인용문(quotes)**: `>`로 시작하는 줄. 답글이 원글을 인용한 부분
- **서명(footers)**: 맨 아래 작성자 서명

### 왜 지워야 하나: 미리 보기 실험
`fetch_20newsgroups(remove=("headers", "footers", "quotes"))`로 세 부분을 지울 수 있습니다. 지우지 않으면 어떻게 될까요? 17–18장에서 배울 분류기를 **미리** 써서 비교해 봅니다. 지금은 코드보다 **숫자 차이**에 집중하세요.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def quick_accuracy(texts, labels) -> float:
    """미리 보기용: TF-IDF + 로지스틱 회귀 정확도 (17–18장에서 직접 만든다)."""
    X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.2,
                                              stratify=labels, random_state=RANDOM_STATE)
    vec = TfidfVectorizer(min_df=2)
    clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(X_tr), y_tr)
    return clf.score(vec.transform(X_te), y_te)


clean_bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                                 remove=("headers", "footers", "quotes"),
                                 shuffle=True, random_state=RANDOM_STATE)
headers_only = [t.split("\n\n", 1)[0] for t in raw_bunch.data]   # 헤더 부분만 (첫 빈 줄 전까지)

print(f"원본 전체 (헤더 포함)   : {quick_accuracy(raw_bunch.data, raw_bunch.target):.3f}")
print(f"헤더만 보고 맞히기      : {quick_accuracy(headers_only, raw_bunch.target):.3f}")
print(f"헤더·서명·인용문 제거   : {quick_accuracy(clean_bunch.data, clean_bunch.target):.3f}")

원본 전체 (헤더 포함)   : 0.938
헤더만 보고 맞히기      : 0.853
헤더·서명·인용문 제거   : 0.838


**본문을 하나도 보지 않고 헤더만으로도** 본문으로 학습한 모델과 비슷하거나 오히려 높은 정확도가 나옵니다. 같은 사람이 같은 게시판에 여러 번 글을 쓰기 때문에, 작성자 주소·소속·제목 표현이 곧 정답 힌트가 됩니다. 이 상태로 만든 모델은 "글의 내용"이 아니라 "누가 썼는지"를 외운 것이라, 과제가 묻는 *"단어 빈도로 문서의 의미를 파악할 수 있는가"*에 답할 수 없습니다.

지우는 이유를 정리하면:
1. **정답 누출 방지**: 헤더의 작성자·소속·제목 정보가 게시판마다 특징적이라, 내용과 상관없이 점수가 부풀려집니다.
2. **개인정보 보호**: 헤더와 서명에 실명과 이메일 주소가 있습니다. 과제의 데이터 윤리 조건에 걸립니다.
3. **중복 정보 제거**: 인용문은 다른 글의 내용이라 같은 문장이 여러 문서에 반복됩니다.

대가도 있습니다. 지운 뒤에는 본문이 비어 버린 글이 생기고(5장에서 116건 확인), 짧은 답글은 주제 단어가 거의 남지 않아 분류가 어려워집니다.

In [4]:
lengths = pd.Series([len(t.split()) for t in clean_bunch.data])
print("제거 후 단어 수 분포:")
print(lengths.describe().round(1).to_string())
print(f"\n단어 10개 미만인 글: {(lengths < 10).sum()}건 ({(lengths < 10).mean():.1%})")

제거 후 단어 수 분포:
count     4941.0
mean       183.0
std        608.9
min          0.0
25%         38.0
50%         77.0
75%        152.0
max      11251.0

단어 10개 미만인 글: 278건 (5.6%)


## 7.4 한국어 데이터: KLUE-YNAT

**KLUE**(Korean Language Understanding Evaluation)는 한국어 언어 모델 평가용 공개 벤치마크이고, 그중 **YNAT**은 연합뉴스 기사 **제목**을 7개 주제로 분류하는 데이터입니다. 라이선스는 **CC BY-SA 4.0**(출처를 밝히면 사용 가능)입니다.

> **NSMC를 쓰지 않은 이유:** 과제 문서가 추천하는 네이버 영화 리뷰(NSMC)는 긍정/부정 감성 데이터라 다음 과제(A4-2, 감성 분석)와 겹칩니다. 주제 분류에는 뉴스 데이터가 더 잘 맞습니다.

GitHub에 공개된 학습 파일(약 44MB)을 `data/raw/`에 내려받아 씁니다. 이미 있으면 다시 받지 않습니다.

In [5]:
import urllib.request
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

if not YNAT_PATH.exists():
    YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("내려받는 중...")
    urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
print("파일:", YNAT_PATH.relative_to(PROJECT), f"{YNAT_PATH.stat().st_size / 1024**2:.1f} MB")

ynat = pd.read_json(YNAT_PATH)
print("shape:", ynat.shape)
print("열   :", ynat.columns.tolist())
ynat[["guid", "title", "label", "predefined_news_category", "date"]].head()

파일: data/raw/ynat-v1.1_train.json 44.0 MB
shape: (45678, 7)
열   : ['guid', 'title', 'predefined_news_category', 'label', 'annotations', 'url', 'date']


,guid,title,label,predefined_news_category,date
0,ynat-v1_train_00000,유튜브 내달 2일까지 크리에이터 지원 공간 운영,생활문화,IT과학,2016.06.30. 오전 10:36
1,ynat-v1_train_00001,어버이날 맑다가 흐려져…남부지방 옅은 황사,생활문화,생활문화,2016.05.08. 오전 5:25
2,ynat-v1_train_00002,내년부터 국가RD 평가 때 논문건수는 반영 않는다,사회,IT과학,2016.03.15. 오후 12:00
3,ynat-v1_train_00003,김명자 신임 과총 회장 원로와 젊은 과학자 지혜 모을 것,사회,IT과학,2017.02.28. 오전 9:54
4,ynat-v1_train_00004,회색인간 작가 김동식 양심고백 등 새 소설집 2권 출간,생활문화,생활문화,2018.04.03. 오전 7:05


| 열 | 내용 |
|---|---|
| `title` | 기사 제목 = **문서** |
| `label` | 주석자 3명이 정한 **주제 = 정답 레이블** |
| `predefined_news_category` | 언론사가 붙인 원래 섹션 |
| `annotations` | 주석자 3명 각각의 판단 |
| `guid` | 문서 고유 ID |

### 레이블은 어떻게 만들어졌나

In [6]:
agree_press = (ynat["label"] == ynat["predefined_news_category"]).mean()
first_choices = ynat["annotations"].map(lambda a: a["annotations"]["first-scope"])
all_agree = first_choices.map(lambda c: len(set(c)) == 1).mean()

print(f"정답 레이블 = 언론사 섹션인 비율 : {agree_press:.1%}")
print(f"주석자 3명의 1순위가 모두 같은 비율: {all_agree:.1%}")
print("\n예시:", ynat.loc[0, "title"])
print("  주석자 1순위:", first_choices[0], "→ 레이블:", ynat.loc[0, "label"])

정답 레이블 = 언론사 섹션인 비율 : 82.9%
주석자 3명의 1순위가 모두 같은 비율: 64.2%

예시: 유튜브 내달 2일까지 크리에이터 지원 공간 운영
  주석자 1순위: ['생활문화', '생활문화', 'IT과학'] → 레이블: 생활문화


**사람끼리도 주제 판단이 상당히 엇갈립니다.** 기사 제목 하나만 보고 "IT과학"인지 "생활문화"인지 가르는 일이 원래 애매하다는 뜻입니다. 그러니 모델 정확도가 100%에 가까울 수 없고, 20장에서 오분류를 분석할 때 "모델이 틀렸다"와 "정답 자체가 애매하다"를 구분해야 합니다.

### 주제 분포와 균등 샘플링

In [7]:
print(ynat["label"].value_counts().to_string())
print(f"\n가장 큰 주제 / 가장 작은 주제 = {ynat['label'].value_counts().max() / ynat['label'].value_counts().min():.2f}배")

label
세계      8320
스포츠     7742
정치      7379
경제      6118
생활문화    5751
IT과학    5235
사회      5133

가장 큰 주제 / 가장 작은 주제 = 1.62배


원본은 주제별로 개수가 다릅니다(세계가 가장 많고 사회가 가장 적음). 그대로 쓰면 모델이 많은 주제 쪽으로 기울고, 4만 5천 건은 TF-IDF 행렬로 만들기에도 큽니다(4장 메모리 계산). 그래서 **주제별로 1,000건씩** 무작위로 뽑아 7,000건을 씁니다. 마지막에 한 번 더 섞어서 주제별로 몰려 있지 않게 합니다.

In [8]:
sampled = (ynat.rename(columns={"title": "text"})[["guid", "text", "label"]]
               .groupby("label", group_keys=False)
               .sample(n=1000, random_state=RANDOM_STATE)
               .sample(frac=1.0, random_state=RANDOM_STATE)     # 전체를 한 번 섞는다
               .reset_index(drop=True))
print(sampled.shape)
print(sampled["label"].value_counts().to_string())
print("\n제목 글자 수:", sampled["text"].str.len().describe()[["mean", "min", "max"]].round(1).to_dict())
sampled.head()

(7000, 3)
label
정치      1000
사회      1000
IT과학    1000
세계      1000
생활문화    1000
스포츠     1000
경제      1000

제목 글자 수: {'mean': 27.2, 'min': 6.0, 'max': 44.0}


,guid,text,label
0,ynat-v1_train_40525,朴대통령 지방·중앙이 힘모아 위기를 기회로,정치
1,ynat-v1_train_43876,셀카·합성영상은 성폭력처벌법 적용 안 돼…보완책 시급,사회
2,ynat-v1_train_22507,아모레퍼시픽 소비자단체와 이해관계자 포럼 열어,사회
3,ynat-v1_train_03274,KT 데이터 공유 앱 Y데이터박스 가입자 10만 돌파,IT과학
4,ynat-v1_train_27146,수괴 사망에도 IS 위협 여전…충성맹세에 말리 테러 배후 자처종합,세계


영어 게시글(평균 수백 단어)과 달리 한국어 제목은 **평균 20~30글자**로 아주 짧습니다. 짧은 문서는 단어가 적어서 BoW로 다루기 더 어렵습니다. 두 데이터의 결과를 비교할 때 이 차이를 기억하세요.

## 7.5 데이터 체크리스트와 개인정보

과제의 데이터 준비 체크리스트를 코드로 점검합니다.

| 항목 | 확인 방법 |
|---|---|
| 500개 이상 문서 | `len(df)` |
| 2개 이상 카테고리 | `df["label"].nunique()` |
| Train/Test 분리 가능 | 카테고리마다 문서가 충분한가 (가장 작은 카테고리 크기) |
| 결측치 제거 완료 | `isna()`, 빈 문자열 |
| 개인정보 미포함 | 이메일·URL·전화번호 패턴 검색 |
| 라이선스 확인 | 데이터 출처 문서 확인 |

### 개인정보 가리기
헤더를 지워도 **본문 안에** 이메일 주소나 URL이 남아 있을 수 있습니다. 검색 결과 스니펫으로 화면에 그대로 나오므로, 불러오는 단계에서 정규식으로 찾아 자리표시자(`<email>`)로 바꿉니다. 정규식은 8장에서 자세히 배웁니다.

In [9]:
_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


print(mask_pii("Contact me at someone@example.com or see http://example.com, tel 02-123-4567."))

for name, pattern in [("이메일", _EMAIL), ("URL", _URL), ("전화번호", _PHONE)]:
    n = sum(bool(pattern.search(t)) for t in clean_bunch.data)
    print(f"헤더 제거 후에도 본문에 {name} 패턴이 있는 20NG 글: {n}건")

Contact me at  <email>  or see  <url>  tel  <phone> .
헤더 제거 후에도 본문에 이메일 패턴이 있는 20NG 글: 724건
헤더 제거 후에도 본문에 URL 패턴이 있는 20NG 글: 1건


헤더 제거 후에도 본문에 전화번호 패턴이 있는 20NG 글: 115건

| 데이터 | 라이선스 | 개인정보 처리 |
|---|---|---|
| 20 Newsgroups | 연구·교육용 공개 데이터 (scikit-learn 배포) | 헤더·서명 제거 + 본문 이메일·URL·전화번호 가리기 |
| KLUE-YNAT | CC BY-SA 4.0 | 기사 제목이라 개인 연락처는 거의 없지만 같은 가리기를 적용 |

### 로딩 함수 완성
지금까지 한 일을 함수로 묶습니다. 5장의 정리 과정(빈 문서, 레이블 충돌 중복, 중복 제거)을 공통 함수 `finalize`로 만들었습니다. **이 셀이 `src/data.py`에 들어갈 코드**이며, 8~10장은 첫 셀에서 이 함수들을 그대로 다시 정의해 씁니다.

In [10]:
def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))


en = load_english()
ko = load_korean()
print("영어:", en.shape, "| 한국어:", ko.shape)

영어: (4797, 2) | 한국어: (7000, 3)


## ✍️ 실습: 데이터 체크리스트 점검 함수

DataFrame(`text`, `label` 열)을 받아 아래 항목을 담은 딕셔너리를 돌려주는 `check_dataset(df)`를 작성하고, `en`과 `ko`에 적용해 표로 출력하세요.

| 키 | 값 |
|---|---|
| `n_docs` | 문서 수 |
| `n_classes` | 카테고리 수 |
| `min_class_size` | 가장 작은 카테고리의 문서 수 |
| `missing_or_empty` | 결측이거나 공백뿐인 본문 수 |
| `duplicates` | 공백 차이를 무시했을 때 중복 본문 수 |
| `pii_hits` | 이메일·URL·전화번호 패턴이 남아 있는 문서 수 (`<email>` 같은 자리표시자는 제외) |
| `passed` | 문서 500개 이상, 카테고리 2개 이상, 나머지 세 항목이 모두 0이면 True |

In [11]:
# 여기에 작성하세요

**정답**

In [12]:
def check_dataset(df: pd.DataFrame) -> dict:
    text = df["text"]
    norm = text.fillna("").str.split().str.join(" ")
    pii = text.fillna("").map(lambda t: any(p.search(t) for p in (_EMAIL, _URL, _PHONE)))
    result = {
        "n_docs": len(df),
        "n_classes": df["label"].nunique(),
        "min_class_size": int(df["label"].value_counts().min()),
        "missing_or_empty": int((text.isna() | (text.fillna("").str.strip() == "")).sum()),
        "duplicates": int(norm.duplicated().sum()),
        "pii_hits": int(pii.sum()),
    }
    result["passed"] = (result["n_docs"] >= 500 and result["n_classes"] >= 2
                        and result["missing_or_empty"] == 0 and result["duplicates"] == 0
                        and result["pii_hits"] == 0)
    return result


pd.DataFrame({"20NG 원본(헤더 제거)": check_dataset(pd.DataFrame({"text": clean_bunch.data, "label": clean_bunch.target})),
              "영어 (load_english)": check_dataset(en),
              "한국어 (load_korean)": check_dataset(ko)})

,20NG 원본(헤더 제거),영어 (load_english),한국어 (load_korean)
n_docs,4941,4797,7000
n_classes,5,5,7
min_class_size,973,953,1000
missing_or_empty,116,0,0
duplicates,140,0,0
pii_hits,757,0,0
passed,False,True,True


`<email>` 자리표시자는 정규식 `\S+@\S+\.\S+`에 걸리지 않으므로 가린 뒤에는 0이 됩니다. 로딩 함수를 거친 두 데이터 모두 체크리스트를 통과합니다.

## 🧐 퀴즈

**Q1.** "the rocket and the team"의 토큰 수와 어휘 수는?
<details><summary>▶ 정답 보기</summary>

토큰은 5개(the, rocket, and, the, team), 어휘는 4개(the, rocket, and, team)입니다.
</details>

**Q2.** 20 Newsgroups에서 헤더를 지우지 않으면 분류 정확도가 높게 나오는데, 왜 이것이 문제인가요?
<details><summary>▶ 정답 보기</summary>

모델이 글의 내용이 아니라 작성자 주소·소속·제목 같은 메타 정보로 게시판을 맞히기 때문입니다. 실제로 측정하려는 "단어로 내용을 파악하는 능력"보다 점수가 부풀려지고, 처음 보는 작성자의 글에는 통하지 않습니다. 개인정보도 함께 들어갑니다.
</details>

**Q3.** KLUE-YNAT에서 주석자 3명의 판단이 모두 같은 비율이 100%보다 한참 낮다는 사실은 모델 평가에 어떤 의미가 있나요?
<details><summary>▶ 정답 보기</summary>

사람도 판단이 갈리는 애매한 문서가 많다는 뜻이므로, 모델 정확도에는 현실적인 상한이 있습니다. 오분류를 분석할 때도 레이블 자체가 애매한 경우를 따로 구분해야 합니다.
</details>

**Q4.** KLUE-YNAT을 원본 그대로 쓰지 않고 주제별 1,000건씩 뽑는 이유 두 가지는?
<details><summary>▶ 정답 보기</summary>

① 주제별 개수가 달라서 모델이 많은 주제로 기우는 것을 막고 지표 해석을 쉽게 하기 위해, ② 4만 5천 건 전체는 NumPy 밀집 행렬로 다루기에 메모리가 너무 크기 때문입니다.
</details>

## ✅ 체크포인트
- [ ] 토큰과 어휘, OOV를 구분해 설명할 수 있다.
- [ ] `remove=("headers", "footers", "quotes")`의 효과를 실험 결과와 함께 말할 수 있다.
- [ ] KLUE-YNAT을 내려받아 주제별로 균등하게 뽑을 수 있다.
- [ ] 본문의 이메일·URL·전화번호를 가릴 수 있다.
- [ ] 데이터 체크리스트를 코드로 점검할 수 있다.

## 🔗 과제 연결과 다음 장
- `load_english`, `load_korean`, `mask_pii`, `finalize`는 `src/data.py`가 됩니다. README의 데이터 설명(출처, 라이선스, 개인정보 처리, 문서 수)에 이 장의 숫자를 씁니다.
- **8장. 텍스트 정제**에서 불러온 본문의 특수문자, 숫자, 대소문자를 어떻게 처리할지 기준을 설계합니다.